<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V1 Dynamic Re-centering Grid + Risk Backtest

This notebook runs `Grid_trading_V1.py` from the current `main` branch. V1 starts from user-defined Initial Floor, Initial Ceiling, Gap, Capital, Recenter Trigger, and Risk Limits. After the run it writes `logs/latest_v1_backtest_log.json` so the latest result can be reviewed directly from GitHub.

## 1. Load current V1 code and mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import base64
import importlib.util

import numpy as np
import pandas as pd
import requests

REPO = 'natdanaiii/Trading'
BRANCH = 'main'
SOURCE_FILE = 'Grid_trading_V1.py'
GITHUB_LOG_PATH = 'logs/latest_v1_backtest_log.json'
LOCAL_SOURCE_PATH = '/content/Grid_trading_V1.py'
LOCAL_LOG_PATH = '/content/latest_v1_backtest_log.json'

source_url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{SOURCE_FILE}'
response = requests.get(source_url, timeout=30)
response.raise_for_status()

with open(LOCAL_SOURCE_PATH, 'w', encoding='utf-8') as f:
    f.write(response.text)

spec = importlib.util.spec_from_file_location('grid_v1', LOCAL_SOURCE_PATH)
v1mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v1mod)

print(f'Loaded current {SOURCE_FILE} from {REPO}/{BRANCH}')

## 2. Configuration and backtest

In [ ]:
df_1m = v1mod.load_market_data(
    v1mod.SYMBOL,
    v1mod.TIMEFRAME,
    v1mod.DATA_DIR,
)

result = v1mod.run_dynamic_grid_backtest(
    df_price=df_1m,
    initial_capital=v1mod.INITIAL_CAPITAL,
    initial_reference=v1mod.INITIAL_REFERENCE,
    gap=v1mod.GRID_GAP,
    number_of_grids=v1mod.NUMBER_OF_GRIDS,
    recenter_trigger_grids=v1mod.RECENTER_TRIGGER_GRIDS,
    buy_fee=v1mod.BUY_FEE,
    sell_fee=v1mod.SELL_FEE,
    min_cash_reserve=v1mod.MIN_CASH_RESERVE,
    max_open_positions=v1mod.MAX_OPEN_POSITIONS,
    max_deployed_capital=v1mod.MAX_DEPLOYED_CAPITAL,
    max_entry_btc_exposure=v1mod.MAX_ENTRY_BTC_EXPOSURE,
    max_drawdown_stop=v1mod.MAX_DRAWDOWN_STOP,
)
summary = result['summary']
audit_checks = v1mod.audit_v1(result)
AUDIT_STATUS = 'PASS' if all(audit_checks.values()) else 'FAIL'

print('===== V1 CONFIGURATION =====')
print(f'Symbol            : {v1mod.SYMBOL}')
print(f'Period            : {v1mod.START_DATE} -> {v1mod.END_DATE}')
print(f'Initial Capital   : {v1mod.INITIAL_CAPITAL:,.2f} USDT')
print(f'Initial Floor     : {v1mod.INITIAL_FLOOR:,.2f} USDT')
print(f'Initial Ceiling   : {v1mod.INITIAL_CEILING:,.2f} USDT')
print(f'Initial Reference : {v1mod.INITIAL_REFERENCE:,.2f} USDT')
print(f'Gap               : {v1mod.GRID_GAP:,.2f} USDT')
print(f'Number of Grids   : {v1mod.NUMBER_OF_GRIDS}')
print(f'Capital / Grid    : {v1mod.CAPITAL_PER_GRID:,.2f} USDT')
print(f'Recenter Trigger  : ±{v1mod.RECENTER_TRIGGER_GRIDS * v1mod.GRID_GAP:,.0f} USDT')
print(f'Min Cash Reserve  : {v1mod.MIN_CASH_RESERVE:,.2f} USDT')
print(f'Max Positions     : {v1mod.MAX_OPEN_POSITIONS}')
print(f'Max Deployed      : {v1mod.MAX_DEPLOYED_CAPITAL:,.2f} USDT')
print(f'Max Entry Exposure: {v1mod.MAX_ENTRY_BTC_EXPOSURE:,.2f} USDT')
print(f'Drawdown BUY Halt : {v1mod.MAX_DRAWDOWN_STOP:.1%}')
print(f'Live Execution    : {v1mod.LIVE_EXECUTION_ENABLED}')

print('\n===== V1 RESULT =====')
print(f'Final Equity      : {summary["final_equity"]:,.2f} USDT')
print(f'Net Return        : {summary["net_return"]:.2%}')
print(f'Annualized        : {summary["annualized_return"]:.2%}')
print(f'Max Drawdown      : {summary["max_drawdown"]:.2%}')
print(f'Calmar Ratio      : {summary["calmar_ratio"]:.3f}')
print(f'Cycles            : {summary["completed_cycles"]:,}')
print(f'Open Positions    : {summary["open_positions"]:,}')
print(f'Final Cash        : {summary["final_cash"]:,.2f} USDT')
print(f'Final BTC         : {summary["final_btc"]:.8f} BTC')
print(f'Recenter Count    : {summary["recenter_count"]:,}')
print(f'Risk Halt         : {summary["risk_halt_triggered"]}')
print(f'Blocked BUYs      : {summary["blocked_buy_counts"]}')

print('\n===== V1 AUDIT =====')
for name, passed in audit_checks.items():
    print(f'{name:32s}: {"PASS" if passed else "FAIL"}')
print(f'Overall                         : {AUDIT_STATUS}')

if AUDIT_STATUS != 'PASS':
    raise AssertionError('V1 AUDIT FAILED')

## 3. Build and upload latest V1 log

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if value is pd.NaT:
        return None
    return value

def records(df):
    return [json_safe(row) for row in df.to_dict(orient='records')]

initial_first_open = float(df_1m.iloc[0]['open'])
initial_inside_range = (
    v1mod.INITIAL_FLOOR <= initial_first_open <= v1mod.INITIAL_CEILING
)

log_payload = {
    'log_schema_version': 1,
    'strategy': 'V1 Dynamic Re-centering Grid + Risk',
    'run_info': {
        'generated_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'repository': REPO,
        'branch': BRANCH,
        'source_file': SOURCE_FILE,
        'notebook': 'Grid_trading_V1.ipynb',
        'symbol': v1mod.SYMBOL,
        'timeframe': v1mod.TIMEFRAME,
        'start_date': v1mod.START_DATE,
        'end_date': v1mod.END_DATE,
        'data_rows': int(len(df_1m)),
        'data_first_time': df_1m['open_time'].min().isoformat(),
        'data_last_time': df_1m['open_time'].max().isoformat(),
    },
    'parameters': {
        'initial_capital': v1mod.INITIAL_CAPITAL,
        'initial_floor': v1mod.INITIAL_FLOOR,
        'initial_ceiling': v1mod.INITIAL_CEILING,
        'gap': v1mod.GRID_GAP,
        'buy_fee': v1mod.BUY_FEE,
        'sell_fee': v1mod.SELL_FEE,
        'recenter_trigger_grids': v1mod.RECENTER_TRIGGER_GRIDS,
        'min_cash_reserve': v1mod.MIN_CASH_RESERVE,
        'max_open_positions': v1mod.MAX_OPEN_POSITIONS,
        'max_deployed_capital': v1mod.MAX_DEPLOYED_CAPITAL,
        'max_entry_btc_exposure': v1mod.MAX_ENTRY_BTC_EXPOSURE,
        'max_drawdown_stop': v1mod.MAX_DRAWDOWN_STOP,
        'live_execution_enabled': v1mod.LIVE_EXECUTION_ENABLED,
    },
    'derived': {
        'number_of_grids': v1mod.NUMBER_OF_GRIDS,
        'initial_reference': v1mod.INITIAL_REFERENCE,
        'capital_per_grid': v1mod.CAPITAL_PER_GRID,
        'recenter_trigger_usdt': v1mod.RECENTER_TRIGGER_GRIDS * v1mod.GRID_GAP,
    },
    'initial_market_diagnostics': {
        'first_open': initial_first_open,
        'first_open_inside_initial_range': bool(initial_inside_range),
        'historical_low': float(df_1m['low'].min()),
        'historical_high': float(df_1m['high'].max()),
    },
    'summary': json_safe(summary),
    'audit': {
        'status': AUDIT_STATUS,
        'checks': json_safe(audit_checks),
    },
    'recenter_diagnostics': {
        'count': int(len(result['recenter_log'])),
        'first_10': records(result['recenter_log'].head(10)),
        'last_10': records(result['recenter_log'].tail(10)),
        'latest_regimes': records(result['regime_history'].tail(10)),
    },
    'risk_diagnostics': {
        'risk_halt_log': records(result['risk_halt_log']),
        'blocked_buy_counts': json_safe(summary['blocked_buy_counts']),
        'min_cash_observed': summary['min_cash_observed'],
        'max_open_positions_observed': summary['max_open_positions_observed'],
        'max_deployed_capital_observed': summary['max_deployed_capital_observed'],
        'max_btc_market_value_observed': summary['max_btc_market_value_observed'],
        'max_entry_btc_exposure_observed': summary['max_entry_btc_exposure_observed'],
    },
}

with open(LOCAL_LOG_PATH, 'w', encoding='utf-8') as f:
    json.dump(log_payload, f, indent=2, allow_nan=False)

print(f'Local log: {LOCAL_LOG_PATH}')

try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None

if not github_token:
    print("GitHub upload SKIPPED: Colab Secret 'GITHUB_TOKEN' was not found.")
else:
    api_url = f'https://api.github.com/repos/{REPO}/contents/{GITHUB_LOG_PATH}'
    headers = {
        'Authorization': f'Bearer {github_token}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    existing = requests.get(api_url, headers=headers, timeout=30)
    body = {
        'message': 'Update latest V1 backtest log',
        'content': base64.b64encode(
            json.dumps(log_payload, indent=2).encode('utf-8')
        ).decode('utf-8'),
        'branch': BRANCH,
    }
    if existing.status_code == 200:
        body['sha'] = existing.json()['sha']

    upload = requests.put(api_url, headers=headers, json=body, timeout=30)
    upload.raise_for_status()
    print('GitHub log upload: SUCCESS')
    print(f'Path             : {GITHUB_LOG_PATH}')
    print(f'Commit SHA       : {upload.json()["commit"]["sha"]}')